# Type I Interferon Response Capacity in PBMCs

Single-cell RNA-seq analysis of pre-therapy PBMC samples to investigate
Type I interferon response capacity and its association with downstream
transcriptional and pathway-level changes.

## Analysis workflow

Raw 10X data → Quality control → Doublet detection → Normalization →
Highly variable genes → Dimensionality reduction → Clustering →
IRC score → Differential expression → Pathway enrichment

# 1. Data Loading and Quality Control

Raw 10X Genomics count matrices from ten samples were loaded individually
and combined into a single AnnData object.

The dataset contains two healthy donor samples (HD1 and HD2) and eight
patient samples (P1–P8).

Quality-control metrics were calculated for:
- number of detected genes
- total UMI counts
- mitochondrial transcript percentage
- hemoglobin transcript percentage

Cells with fewer than 300 detected genes, mitochondrial content ≥15%,
or hemoglobin content ≥5% were removed.

Because unusually high gene and UMI counts can indicate potential
doublets, Scrublet was subsequently applied independently to each sample.
Cells predicted as doublets were removed before downstream analysis.

In [45]:
# ============================================================
# 1. DATA LOADING, QUALITY CONTROL & DOUBLET DETECTION
# ============================================================

import scanpy as sc
import anndata as ad
import scrublet as scr

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from importlib.metadata import version


# ------------------------------------------------------------
# Reproducibility and plotting settings
# ------------------------------------------------------------

sc.settings.verbosity = 1

sc.set_figure_params(
    dpi=100,
    dpi_save=300,
    figsize=(6, 4)
)


# ------------------------------------------------------------
# Define directories
# ------------------------------------------------------------

data_dir = Path("../data")

results_dir = Path("../results")
figures_dir = results_dir / "figures"
preprocessing_dir = results_dir / "preprocessing_tables"

figures_dir.mkdir(parents=True, exist_ok=True)
preprocessing_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Software versions
# ------------------------------------------------------------

print("Scanpy version:", sc.__version__)
print("AnnData version:", version("anndata"))
print("Scrublet version:", version("scrublet"))

Scanpy version: 1.11.5
AnnData version: 0.12.19
Scrublet version: 0.2.3


/tmp/ipykernel_15926/1711777610.py:48: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("Scanpy version:", sc.__version__)


## 1.1 Load raw 10X data

Each sample is loaded separately so that sample-level metadata can be
retained and Scrublet can later be applied independently to each sample.

In [47]:
# ------------------------------------------------------------
# Identify sample directories
# ------------------------------------------------------------

sample_dirs = sorted([
    path for path in data_dir.iterdir()
    if path.is_dir()
])

print("Samples found:")

for path in sample_dirs:
    print(" -", path.name)

Samples found:
 - HD1
 - HD2
 - P1
 - P2
 - P3
 - P4
 - P5
 - P6
 - P7
 - P8


In [48]:
# ------------------------------------------------------------
# Load each 10X dataset
# ------------------------------------------------------------

adatas = []

for sample_path in sample_dirs:

    sample_id = sample_path.name

    print(f"\nLoading {sample_id}...")

    sample_adata = sc.read_10x_mtx(
        sample_path,
        var_names="gene_symbols",
        cache=False
    )

    sample_adata.var_names_make_unique()

    # Store sample information
    sample_adata.obs["sample"] = sample_id

    # Classify donor
    if sample_id.startswith("HD"):
        sample_adata.obs["donor_type"] = "Healthy donor"
    else:
        sample_adata.obs["donor_type"] = "Patient"

    adatas.append(sample_adata)

    print(
        f"{sample_id}: "
        f"{sample_adata.n_obs} cells × "
        f"{sample_adata.n_vars} genes"
    )


Loading HD1...
HD1: 7565 cells × 36601 genes

Loading HD2...
HD2: 11952 cells × 36601 genes

Loading P1...
P1: 10845 cells × 36601 genes

Loading P2...
P2: 14493 cells × 36601 genes

Loading P3...
P3: 13010 cells × 36601 genes

Loading P4...
P4: 10787 cells × 36601 genes

Loading P5...
P5: 2992 cells × 36601 genes

Loading P6...
P6: 9948 cells × 36601 genes

Loading P7...
P7: 9704 cells × 36601 genes

Loading P8...
P8: 9724 cells × 36601 genes


In [49]:
# ------------------------------------------------------------
# Merge all samples
# ------------------------------------------------------------

adata = ad.concat(
    adatas,
    join="outer",
    label="batch",
    keys=[x.obs["sample"].iloc[0] for x in adatas],
    index_unique="-"
)

adata.var_names_make_unique()

print("\nMerged dataset:")
print(adata)


Merged dataset:
AnnData object with n_obs × n_vars = 101020 × 36601
    obs: 'sample', 'donor_type', 'batch'


In [50]:
n_cells_raw = adata.n_obs
n_genes_raw = adata.n_vars

In [51]:
# ------------------------------------------------------------
# Save merged raw dataset
# ------------------------------------------------------------

adata.write(
    preprocessing_dir / "merged_raw.h5ad"
)

print(
    "Saved:",
    preprocessing_dir / "merged_raw.h5ad"
)

Saved: ../results/preprocessing_tables/merged_raw.h5ad


## 1.2 Calculate quality-control metrics

Mitochondrial, ribosomal, and hemoglobin genes were annotated to allow
calculation of their relative contribution to each cell's transcript
counts.

In [52]:
# ------------------------------------------------------------
# Define QC gene categories
# ------------------------------------------------------------

adata.var["mt"] = (
    adata.var_names.str.upper().str.startswith("MT-")
)

adata.var["ribo"] = (
    adata.var_names.str.upper().str.startswith("RPS")
    | adata.var_names.str.upper().str.startswith("RPL")
)

adata.var["hb"] = (
    adata.var_names.str.upper().str.startswith("HBA")
    | adata.var_names.str.upper().str.startswith("HBB")
    | adata.var_names.str.upper().str.startswith("HBD")
    | adata.var_names.str.upper().str.startswith("HBE")
    | adata.var_names.str.upper().str.startswith("HBG")
    | adata.var_names.str.upper().str.startswith("HBM")
    | adata.var_names.str.upper().str.startswith("HBQ")
    | adata.var_names.str.upper().str.startswith("HBZ")
)


# ------------------------------------------------------------
# Calculate QC metrics
# ------------------------------------------------------------

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    inplace=True
)

In [53]:
adata

AnnData object with n_obs × n_vars = 101020 × 36601
    obs: 'sample', 'donor_type', 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [54]:
adata.obs[
    [
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "pct_counts_hb"
    ]
].head()

,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_ribo,pct_counts_hb
AAACAGCCAAACTCAT-1-HD1,1982,4498.0,6.313917,2.178746,0.022232
AAACAGCCAATTAACC-1-HD1,2944,8411.0,5.956485,2.544287,0.023778
AAACAGCCACAATACT-1-HD1,2564,7222.0,6.424813,3.406259,0.013847
AAACAGCCACTAAGCC-1-HD1,2451,6700.0,6.492537,1.582090,0.059701
AAACAGCCAGATAGAC-1-HD1,1877,3910.0,4.373402,8.900256,0.000000


## 1.3 QC before filtering

QC distributions were examined across samples before filtering.
This provides a baseline for assessing the effect of the subsequent
quality-control and doublet-removal steps.

In [55]:
# ------------------------------------------------------------
# QC violin plots before filtering by sample
# ------------------------------------------------------------

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_hb"],
    groupby="sample",
    jitter=0.4,
    multi_panel=True,
    show=False
)

plt.savefig(
    figures_dir / "01_qc_before_filtering_by_sample.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [56]:
# ------------------------------------------------------------
# QC violin plots before filtering by donor type
# ------------------------------------------------------------

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_hb"],
    groupby="donor_type",
    jitter=0.4,
    multi_panel=True,
    show=False
)

plt.savefig(
    figures_dir / "02_qc_before_filtering_by_donor_type.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [57]:
# ------------------------------------------------------------
# QC summary by sample
# ------------------------------------------------------------

qc_summary = (
    adata.obs
    .groupby("sample")
    [
        [
            "n_genes_by_counts",
            "total_counts",
            "pct_counts_mt",
            "pct_counts_hb"
        ]
    ]
    .median()
)

qc_summary.to_csv(
    preprocessing_dir /
    "qc_summary_before_filtering_by_sample.csv"
)

display(qc_summary)

/tmp/ipykernel_15926/2765177528.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("sample")


,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_hb
sample,,,,
HD1,2033.0,4412.0,5.821004,0.0
HD2,2068.0,4194.0,4.636642,0.0
P1,2245.0,5049.0,6.135209,0.0
P2,1909.0,4022.0,5.011136,0.0
P3,1996.5,4317.0,6.621499,0.0
P4,1984.0,4513.0,5.272609,0.0
P5,1298.0,2545.0,11.416914,0.0
P6,1678.0,3454.5,10.859177,0.0
P7,1960.5,3965.5,8.808832,0.0


In [58]:
# ------------------------------------------------------------
# QC summary by donor type
# ------------------------------------------------------------

qc_summary = (
    adata.obs
    .groupby("donor_type")
    [
        [
            "n_genes_by_counts",
            "total_counts",
            "pct_counts_mt",
            "pct_counts_hb"
        ]
    ]
    .median()
)

qc_summary.to_csv(
    preprocessing_dir /
    "qc_summary_before_filtering_by_donor_type.csv"
)

display(qc_summary)

/tmp/ipykernel_15926/2678104160.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("donor_type")


,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_hb
donor_type,,,,
Healthy donor,2055.0,4293.0,5.049551,0.0
Patient,1946.0,4189.0,7.076102,0.0


## 1.4 Basic QC filtering

Cells with fewer than 300 detected genes were removed as low-complexity
cells. Cells with mitochondrial transcript percentages ≥15% or
hemoglobin transcript percentages ≥5% were also excluded.

No arbitrary upper threshold was imposed on total UMI counts or number
of detected genes because unusually high values were investigated using
the subsequent doublet-detection step.

In [59]:
# ------------------------------------------------------------
# Record initial cell number
# ------------------------------------------------------------

n_cells_before_basic_qc = adata.n_obs

print(
    f"Cells before basic QC: "
    f"{n_cells_before_basic_qc:,}"
)


# ------------------------------------------------------------
# Minimum detected genes
# ------------------------------------------------------------

sc.pp.filter_cells(
    adata,
    min_genes=300
)

print(
    f"After minimum gene filter: "
    f"{adata.n_obs:,} cells"
)


# ------------------------------------------------------------
# Mitochondrial filtering
# ------------------------------------------------------------

adata = adata[
    adata.obs["pct_counts_mt"] < 15,
    :
].copy()

print(
    f"After mitochondrial filter: "
    f"{adata.n_obs:,} cells"
)


# ------------------------------------------------------------
# Hemoglobin filtering
# ------------------------------------------------------------

adata = adata[
    adata.obs["pct_counts_hb"] < 5,
    :
].copy()

print(
    f"After hemoglobin filter: "
    f"{adata.n_obs:,} cells"
)

Cells before basic QC: 101,020
After minimum gene filter: 98,826 cells
After mitochondrial filter: 94,514 cells
After hemoglobin filter: 94,513 cells


In [60]:
n_cells_after_basic_qc = adata.n_obs
n_genes_after_basic_qc = adata.n_vars

## 1.5 Doublet detection using Scrublet

Potential doublets were detected using Scrublet. Doublet detection was
performed separately for each sample because doublet rates and
transcriptional distributions can differ between libraries.

Scrublet assigns each cell a doublet score and predicts whether the cell
is a doublet based on simulated doublets generated from the observed
expression data.

In [61]:
# ------------------------------------------------------------
# Initialize Scrublet result columns
# ------------------------------------------------------------

adata.obs["doublet_score"] = 0.0
adata.obs["predicted_doublet"] = False


# ------------------------------------------------------------
# Run Scrublet independently for each sample
# ------------------------------------------------------------

for sample_id in adata.obs["sample"].unique():

    print(f"\nRunning Scrublet for {sample_id}...")

    sample_mask = (
        adata.obs["sample"] == sample_id
    )

    counts_matrix = adata[sample_mask].X

    scrub = scr.Scrublet(
        counts_matrix
    )

    doublet_scores, predicted_doublets = (
        scrub.scrub_doublets(
            verbose=False
        )
    )

    adata.obs.loc[
        sample_mask,
        "doublet_score"
    ] = doublet_scores

    adata.obs.loc[
        sample_mask,
        "predicted_doublet"
    ] = predicted_doublets


Running Scrublet for HD1...

Running Scrublet for HD2...

Running Scrublet for P1...

Running Scrublet for P2...

Running Scrublet for P3...

Running Scrublet for P4...

Running Scrublet for P5...

Running Scrublet for P6...

Running Scrublet for P7...

Running Scrublet for P8...


In [62]:
# ------------------------------------------------------------
# Human-readable labels
# ------------------------------------------------------------

adata.obs["doublet_info"] = (
    adata.obs["predicted_doublet"]
    .map({
        False: "Singlet",
        True: "Doublet"
    })
)


# ------------------------------------------------------------
# Overall doublet counts
# ------------------------------------------------------------

doublet_counts = (
    adata.obs["predicted_doublet"]
    .value_counts()
)

print("Scrublet results:")
print(doublet_counts)


# ------------------------------------------------------------
# Overall doublet rate
# ------------------------------------------------------------

doublet_rate = (
    adata.obs["predicted_doublet"].mean()
    * 100
)

print(
    f"\nOverall predicted doublet rate: "
    f"{doublet_rate:.2f}%"
)

Scrublet results:
predicted_doublet
False    87448
True      7065
Name: count, dtype: int64

Overall predicted doublet rate: 7.48%


## 1.6 Inspect predicted doublets

The relationship between Scrublet predictions and common QC metrics was
examined. Doublets are expected to show, in many cases, elevated numbers
of detected genes and total UMI counts.

In [63]:
sc.pl.violin(
    adata,
    "n_genes_by_counts",
    groupby="doublet_info",
    jitter=0.4,
    show=False
)

plt.savefig(
    figures_dir / "03_doublet_genes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [64]:
sc.pl.violin(
    adata,
    "total_counts",
    groupby="doublet_info",
    jitter=0.4,
    show=False
)

plt.savefig(
    figures_dir / "04_doublet_total_counts.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [65]:
sc.pl.violin(
    adata,
    "doublet_score",
    groupby="sample",
    jitter=0.4,
    rotation=45,
    show=False
)

plt.savefig(
    figures_dir / "05_scrublet_scores_by_sample.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [66]:
# ------------------------------------------------------------
# Per-sample doublet summary
# ------------------------------------------------------------

doublet_summary = (
    adata.obs
    .groupby("sample")
    .agg(
        cells=("sample", "size"),
        predicted_doublets=(
            "predicted_doublet",
            "sum"
        ),
        median_doublet_score=(
            "doublet_score",
            "median"
        )
    )
)

doublet_summary["doublet_rate_percent"] = (
    doublet_summary["predicted_doublets"]
    / doublet_summary["cells"]
    * 100
)

doublet_summary.to_csv(
    preprocessing_dir /
    "doublet_summary_by_sample.csv"
)

display(doublet_summary)

/tmp/ipykernel_15926/4218823088.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("sample")


,cells,predicted_doublets,median_doublet_score,doublet_rate_percent
sample,,,,
HD1,7482,361,0.067273,4.824913
HD2,11694,834,0.076205,7.131862
P1,9718,810,0.065303,8.335048
P2,13774,1482,0.071527,10.759402
P3,12590,949,0.065744,7.537728
P4,10194,779,0.058140,7.641750
P5,2357,80,0.083636,3.394145
P6,8644,553,0.057076,6.397501
P7,8850,585,0.057737,6.610169


## 1.7 Remove Scrublet-predicted doublets

Cells classified as doublets by Scrublet were excluded from the
downstream analysis. These cells are referred to as
"Scrublet-predicted doublets" because doublet status is computationally
inferred rather than experimentally confirmed.

In [67]:
# ------------------------------------------------------------
# Record numbers before removal
# ------------------------------------------------------------

n_cells_before_doublet_removal = adata.n_obs

n_doublets = (
    adata.obs["predicted_doublet"].sum()
)

print(
    f"Cells before doublet removal: "
    f"{n_cells_before_doublet_removal:,}"
)

print(
    f"Scrublet-predicted doublets: "
    f"{n_doublets:,}"
)


# ------------------------------------------------------------
# Remove predicted doublets
# ------------------------------------------------------------

adata = adata[
    adata.obs["predicted_doublet"] == False,
    :
].copy()

print(
    f"Cells after doublet removal: "
    f"{adata.n_obs:,}"
)

Cells before doublet removal: 94,513
Scrublet-predicted doublets: 7,065
Cells after doublet removal: 87,448


In [68]:
n_cells_after_scrublet = adata.n_obs
n_genes_after_scrublet = adata.n_vars

In [28]:
# ------------------------------------------------------------
# Remove genes detected in fewer than 5 cells
# ------------------------------------------------------------

n_genes_before_filtering = adata.n_vars

sc.pp.filter_genes(
    adata,
    min_cells=5
)

n_genes_after_filtering = adata.n_vars

print(
    f"Genes before filtering: "
    f"{n_genes_before_filtering:,}"
)

print(
    f"Genes after filtering: "
    f"{n_genes_after_filtering:,}"
)

Genes before filtering: 36,601
Genes after filtering: 30,744


## 1.8 QC after filtering

QC metrics were visualized again after basic QC filtering, Scrublet
doublet removal, and gene filtering.

The post-filtering distributions provide a direct assessment of whether
the applied QC procedure successfully removed low-quality cells and
extreme observations.

In [69]:
# ------------------------------------------------------------
# Recalculate QC metrics
# ------------------------------------------------------------

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    inplace=True
)

In [70]:
adata.obs[
    [
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "pct_counts_hb"
    ]
].head()

,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_ribo,pct_counts_hb
AAACAGCCAAACTCAT-1-HD1,1982,4498.0,6.313917,2.178746,0.022232
AAACAGCCAATTAACC-1-HD1,2944,8411.0,5.956485,2.544287,0.023778
AAACAGCCACAATACT-1-HD1,2564,7222.0,6.424813,3.406259,0.013847
AAACAGCCACTAAGCC-1-HD1,2451,6700.0,6.492537,1.582090,0.059701
AAACAGCCAGATAGAC-1-HD1,1877,3910.0,4.373402,8.900256,0.000000


In [71]:
# ------------------------------------------------------------
# QC violin plots after filtering by sample
# ------------------------------------------------------------
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_hb"],
    groupby="sample",
    jitter=0.4,
    multi_panel=True,
    show=False
)

plt.savefig(
    figures_dir / "06_qc_after_filtering_by_sample.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [72]:
# ------------------------------------------------------------
# QC violin plots after filtering by donor type
# ------------------------------------------------------------
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_hb"],
    groupby="donor_type",
    jitter=0.4,
    multi_panel=True,
    show=False
)

plt.savefig(
    figures_dir / "07_qc_after_filtering_by_donor_type.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

In [73]:
# ------------------------------------------------------------
# Final cell counts by sample
# ------------------------------------------------------------

final_sample_counts = (
    adata.obs["sample"]
    .value_counts()
    .sort_index()
)

final_sample_counts.to_csv(
    preprocessing_dir /
    "final_cell_counts_by_sample.csv"
)

print("Final cell counts by sample:")
print(final_sample_counts)

Final cell counts by sample:
sample
HD1     7121
HD2    10860
P1      8908
P2     12292
P3     11641
P4      9415
P5      2277
P6      8091
P7      8265
P8      8578
Name: count, dtype: int64


In [74]:
# ------------------------------------------------------------
# Final QC summary by sample
# ------------------------------------------------------------

final_qc_summary = (
    adata.obs
    .groupby("sample")
    [
        [
            "n_genes_by_counts",
            "total_counts",
            "pct_counts_mt",
            "pct_counts_hb"
        ]
    ]
    .median()
)

final_qc_summary.to_csv(
    preprocessing_dir /
    "qc_summary_after_filtering_by_sample.csv"
)

display(final_qc_summary)

/tmp/ipykernel_15926/3986880531.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("sample")


,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_hb
sample,,,,
HD1,2011.0,4338.0,5.796440,0.0
HD2,2041.0,4112.0,4.589397,0.0
P1,2263.0,5120.5,5.743270,0.0
P2,1922.0,4078.5,4.941286,0.0
P3,1968.0,4228.0,6.537983,0.0
P4,1977.0,4492.0,5.123043,0.0
P5,1336.0,2616.0,10.539737,0.0
P6,1677.0,3426.0,10.356200,0.0
P7,1964.0,3966.0,8.463251,0.0


In [75]:
# ------------------------------------------------------------
# Final QC summary by donor type
# ------------------------------------------------------------

final_qc_summary = (
    adata.obs
    .groupby("donor_type")
    [
        [
            "n_genes_by_counts",
            "total_counts",
            "pct_counts_mt",
            "pct_counts_hb"
        ]
    ]
    .median()
)

final_qc_summary.to_csv(
    preprocessing_dir /
    "qc_summary_after_filtering_by_donor_type.csv"
)

display(final_qc_summary)

/tmp/ipykernel_15926/1438885814.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("donor_type")


,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_hb
donor_type,,,,
Healthy donor,2030.0,4202.0,5.018146,0.0
Patient,1952.0,4205.0,6.804375,0.0


In [76]:
# ------------------------------------------------------------
# Filtering summary: cells and genes
# ------------------------------------------------------------

filtering_summary = pd.DataFrame({
    "stage": [
        "Raw merged data",
        "After basic QC",
        "After Scrublet doublet removal",
        "After gene filtering"
    ],
    "cells": [
        n_cells_raw,
        n_cells_after_basic_qc,
        n_cells_after_scrublet,
        adata.n_obs
    ],
    "genes": [
        n_genes_raw,
        n_genes_after_basic_qc,
        n_genes_after_scrublet,
        n_genes_after_filtering
    ]
})

filtering_summary.to_csv(
    preprocessing_dir /
    "filtering_summary.csv",
    index=False
)

display(filtering_summary)

,stage,cells,genes
0,Raw merged data,101020,36601
1,After basic QC,94513,36601
2,After Scrublet doublet removal,87448,36601
3,After gene filtering,87448,30744


## 1.9 Save final QC-filtered dataset

The final QC-filtered AnnData object was saved and will serve as the
input for all subsequent normalization, dimensionality-reduction,
IRC-score, differential-expression, and pathway-enrichment analyses.

In [77]:
# ------------------------------------------------------------
# Save final QC-filtered AnnData
# ------------------------------------------------------------

adata.write(
    preprocessing_dir /
    "merged_qc_filtered.h5ad"
)

print(
    "Final dataset saved to:"
)

print(
    preprocessing_dir /
    "merged_qc_filtered.h5ad"
)


# ------------------------------------------------------------
# Final dataset dimensions
# ------------------------------------------------------------

print("\n===================================")
print("FINAL DATASET")
print("===================================")

print(
    f"Cells: {adata.n_obs:,}"
)

print(
    f"Genes: {adata.n_vars:,}"
)

print(
    f"Samples: {adata.obs['sample'].nunique()}"
)

print("\nWorkflow complete.")

Final dataset saved to:
../results/preprocessing_tables/merged_qc_filtered.h5ad

FINAL DATASET
Cells: 87,448
Genes: 36,601
Samples: 10

Workflow complete.


In [78]:
adata

AnnData object with n_obs × n_vars = 87448 × 36601
    obs: 'sample', 'donor_type', 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'doublet_info'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'sample_colors', 'donor_type_colors', 'doublet_info_colors'

# Section 1 output summary

| Output                                          | Meaning                                                                          |
| ----------------------------------------------- | -------------------------------------------------------------------------------- |
| `merged_raw.h5ad`                               | Raw merged AnnData object containing all 10 samples before QC filtering          |
| `01_qc_before_filtering_by_sample.png`          | QC metrics visualized for individual samples before filtering                    |
| `02_qc_before_filtering_by_donor_type.png`      | QC metrics visualized by donor type before filtering                             |
| `qc_summary_before_filtering_by_sample.csv`     | Sample-level QC statistics before filtering                                      |
| `qc_summary_before_filtering_by_donor_type.csv` | Donor-type-level QC statistics before filtering                                  |
| `03_doublet_genes.png`                          | Comparison of detected genes between Scrublet-predicted singlets and doublets    |
| `04_doublet_total_counts.png`                   | Comparison of total counts between Scrublet-predicted singlets and doublets      |
| `05_scrublet_scores_by_sample.png`              | Distribution of Scrublet doublet scores across samples                           |
| `doublet_summary_by_sample.csv`                 | Sample-level summary of Scrublet-predicted doublets                              |
| `06_qc_after_filtering_by_sample.png`           | QC metrics visualized for individual samples after filtering and doublet removal |
| `07_qc_after_filtering_by_donor_type.png`       | QC metrics visualized by donor type after filtering and doublet removal          |
| `qc_summary_after_filtering_by_sample.csv`      | Sample-level QC statistics after filtering                                       |
| `qc_summary_after_filtering_by_donor_type.csv`  | Donor-type-level QC statistics after filtering                                   |
| `final_cell_counts_by_sample.csv`               | Final number of retained cells from each sample                                  |
| `filtering_summary.csv`                         | Audit trail showing cell and gene counts at each preprocessing stage             |
| `merged_qc_filtered.h5ad`                       | Final QC-filtered AnnData object used as the input for Section 2                 |

## Interpretation

Section 1 establishes **which cells and genes should be trusted for downstream analysis**.

The merged raw dataset is first evaluated using standard quality-control metrics, including detected genes, total counts, mitochondrial RNA, ribosomal RNA, and hemoglobin RNA.

Low-quality cells are removed using the predefined QC thresholds, followed by **Scrublet-based doublet detection** to identify and remove cells predicted to represent multiplets. Genes detected in fewer than five cells are then removed.

The resulting dataset,

`merged_qc_filtered.h5ad`

contains the final set of cells and genes retained after quality control and becomes the input to **Section 2: Normalization and Highly Variable Gene Selection**.
